In [46]:
from seleniumbase import Driver
from selenium.webdriver.common.by import By
import pandas as pd
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [47]:
driver = Driver(uc=True)   # undetected-chromedriver

In [ ]:
# BASE_URL = 'https://www.elmenus.com/alexandria/abo-rawia-loag/semouha-p9qw/reviews'

In [48]:
urls = {'Abo Rawia':'https://www.elmenus.com/alexandria/abo-rawia-loag/semouha-p9qw/reviews',
        'Sultan Ayub':'https://www.elmenus.com/alexandria/sultan-ayub-5kg8/semouha-6gyo/reviews',
        "Dixi's Fried Chicken":'https://www.elmenus.com/alexandria/dixis-fried-chicken-p6kxr/moustafa-kamel-z75am/reviews',
        '5 Roosters':'https://www.elmenus.com/alexandria/5-roosters-8wxqw/al-azarita-8z663/reviews',
        'Hadramout':'https://www.elmenus.com/alexandria/hadramout-doakx/al-ibrahimia-my3mo/reviews',
        'Soori':'https://www.elmenus.com/alexandria/soori-w5r7/semouha-x8525/reviews'}

In [ ]:
# driver.get(BASE_URL)

In [ ]:
# review = driver.find_elements(By.XPATH,'//div[contains(@class,"review__comment")]//p')
# for rev in review:
#     print(rev.text)

In [51]:
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
def extract_elmenu_reviews(driver, url, max_reviews=10000):

    driver.get(url)
    time.sleep(3)

    reviews_data = []
    seen_comments = set()
    old_len = 0
    count = 0
    while True:
        # wait for reviews to be present
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located(
                (By.XPATH, '//div[contains(@class,"review__comment")]//p')
            )
        )

        comments = driver.find_elements(By.XPATH, '//div[contains(@class,"review__comment")]//p')

        for c in comments:
            text = c.text.strip()

            if not text:
                continue

            if text in seen_comments:
                continue

            seen_comments.add(text)
            reviews_data.append({"comment": text})

        # print(f"Collected {len(reviews_data)} comments")
        if len(reviews_data) == old_len:
            count+=1
            
        if len(reviews_data) == old_len and count>=3:
            break
        if len(reviews_data) >= max_reviews:
            break
        old_len = len(reviews_data)
        # try clicking Load More button
        try:
            load_more_btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, 'button.btn.btn-primary.btn--load-more')
                )
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", load_more_btn)
            time.sleep(1)
            driver.execute_script("arguments[0].click();", load_more_btn)
            time.sleep(2)

        except (NoSuchElementException, ElementClickInterceptedException):
            print("No more Load More button. Finished.")
            break

    df = pd.DataFrame(reviews_data)
    return df


In [52]:
for key,value in urls.items():
    df = extract_elmenu_reviews(driver,value)
    df.to_csv(f'elmenu_{key}_resturant.csv')
    print(f'{key} has been scraped we got {df.shape}')

Abo Rawia has been scraped we got (605, 1)
Sultan Ayub has been scraped we got (949, 1)
Dixi's Fried Chicken has been scraped we got (986, 1)
5 Roosters has been scraped we got (588, 1)
Hadramout has been scraped we got (611, 1)
Soori has been scraped we got (673, 1)


In [ ]:
# df.duplicated().sum()

np.int64(0)

In [ ]:
# df.to_csv('elmenu__resturant.csv')